## imports

In [1]:
import copy
import datetime
import glob
import itertools
import json
import logging
import os
import pickle
import re
import sys
import time
import warnings
import traceback
import math 
from functools import partial
from pathlib import Path
from typing import Callable, Literal, Union

import numpy as np
import pandas as pd
import xarray as xr
from netCDF4 import Dataset
from scipy import stats
from scipy.ndimage import center_of_mass
from scipy.spatial import KDTree
import skimage
import cfgrib
import fsspec

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as patches
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.ticker as plticker
from matplotlib import cm
from matplotlib.animation import FuncAnimation
from matplotlib.colors import ListedColormap, TwoSlopeNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER
import seaborn as sns
import turbo_colormap

from itertools import product
import pytz
from tqdm.notebook import tqdm
from IPython.display import HTML, display

import dask
import dask.bag
from dask import delayed
from dask.diagnostics import CacheProfiler, Profiler, ProgressBar, ResourceProfiler
from dask.distributed import Client, LocalCluster, get_worker, as_completed
from dask_jobqueue import SLURMCluster

import tobac
import tobac.merge_split as ms
from tobac.utils import decorators


/home/rauth/miniforge3/envs/research/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (
/home/rauth/miniforge3/envs/research/lib/python3.12/site-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.42.0 or higher is recommended. You are running version 2.41.0
  warnings.warn(


In [2]:
def fill_missing_timestamps(df):

    #Fill missing timestamps per cell by interpolating rows for missing times. 
    # Only updates 'frame', 'timestr', and 'time_cell' for new rows.

    logging.info("Sorting dataframe...")
    t0 = time.perf_counter()
    
    df = df.sort_values(["cell", "time"]).reset_index(drop=True)
    
    logging.info(
        f"Sort complete ({time.perf_counter()-t0:.1f} s)"
    )

    # Unique times across entire DataFrame
    unique_times = pd.Series(sorted(df['time'].unique()))
    logging.info(f"{len(unique_times):,} unique timestamps")

    # Prepare list to collect filled data per cell
    filled_dfs = []

    groups = df.groupby("cell")
    logging.info(f"{groups.ngroups:,} cell groups")
    #for cell, group in df.groupby('cell'):
    for i, (cell, group) in enumerate(groups):
        if i % 1000 == 0:
            logging.info(
                f"Processed {i:,}/{groups.ngroups:,} groups"
            )
        if cell <= 0:  # skip untracked or invalid cells
            continue
        group = group.reset_index(drop=True)
        # All unique times for this cell
        times_cell = group['time']
        # Construct a full timeline for this cell 
        full_times = unique_times[(unique_times >= times_cell.min()) & (unique_times <= times_cell.max())]
        logging.info(f"full_times = {len(full_times):,}")
        # Identify missing times for this cell
        missing_times = full_times[~full_times.isin(times_cell)]
        logging.info(f"missing_times = {len(missing_times):,}")
        if missing_times.empty:
            # No missing times for this cell, skip interpolation and proceed to next cell
            filled_dfs.append(group)
            continue 
        # For each missing time, create an interpolated row
        interpolated_rows = []
        for missing_time in missing_times:
            # Find idx where to insert
            after_idx = group[group['time'] < missing_time].index.max()
            before_idx = group[group['time'] > missing_time].index.min()

            # If no before or after found, skip (shouldn't happen)
            if pd.isna(after_idx) or pd.isna(before_idx):
                continue

            # Get rows before and after missing_time
            row_before = group.loc[after_idx]
            row_after = group.loc[before_idx]

            # Average latitude, longitude, hdim_1, hdim_2
            lat_interp = (row_before['latitude'] + row_after['latitude']) / 2
            lon_interp = (row_before['longitude'] + row_after['longitude']) / 2
            hdim1_interp = (row_before['hdim_1'] + row_after['hdim_1']) / 2
            hdim2_interp = (row_before['hdim_2'] + row_after['hdim_2']) / 2

            # Copy row_before and update fields
            new_row = row_before.copy()
            new_row['time'] = missing_time
            new_row['latitude'] = lat_interp
            new_row['longitude'] = lon_interp
            new_row['hdim_1'] = hdim1_interp
            new_row['hdim_2'] = hdim2_interp

            new_row['time_cell'] = row_before['time_cell'] + pd.Timedelta(seconds=60)

            # Other columns stay as in row_before 
            interpolated_rows.append(new_row)

        # Combine original and interpolated rows
        group_full = pd.concat([group, pd.DataFrame(interpolated_rows)], ignore_index=True)

        # Sort by time
        group_full = group_full.sort_values('time').reset_index(drop=True)

        filled_dfs.append(group_full)

    # Combine all cells back
    df_full = pd.concat(filled_dfs, ignore_index=True)

    # Mark interpolated rows by checking if original (cell, time) exists
    original_index = df.set_index(['cell', 'time']).index
    df_full_index = df_full.set_index(['cell', 'time']).index
    df_full['is_interpolated'] = ~df_full_index.isin(original_index)

    # Only recalc time-dependent fields for interpolated rows
    interp_rows = df_full['is_interpolated']

    # timestr as formatted string
    df_full.loc[interp_rows, 'timestr'] = df_full.loc[interp_rows, 'time'].dt.strftime('%Y-%m-%d %H:%M:%S')

    # Recalculate frame only for interpolated rows:
    unique_times_sorted = sorted(df['time'].unique())
    time_to_frame = {t: i for i, t in enumerate(unique_times_sorted)}
    df_full.loc[interp_rows, 'frame'] = df_full.loc[interp_rows, 'time'].map(time_to_frame)

    # Drop helper column
    #df_full.drop(columns=['is_interpolated'], inplace=True)

    # Reset index before returning
    df_full = df_full.reset_index(drop=True)
    
    # Renumber feature
    df_full['feature'] = range(len(df_full))

    return df_full


In [4]:
def track_feature_families(in_feat_arr: pd.DataFrame, in_family_stat_arr: pd.DataFrame, 
                           maintain_family_metric:Literal["cells", "area"] ='area', 
                           feat_family_column_name: str = "feature_family_id", 
                           cell_column_name:str = "cell", tracked_family_column_name:str = "tracked_family_id"):

    
    time_col_name: str = 'time'

    family_ids_of_interest = []#[139140, 139055]

    # number to start the new tracking ID with
    curr_tracked_family_num = 1

    # the fraction of previous family cells to retain to be considered part of the same track
    maintain_track_cell_fraction = 0.5

    # the fraction of previous family area to retain to be considered part of the same track
    maintain_track_area_fraction = 0.3

    if maintain_track_cell_fraction <0.5:
        raise ValueError("Cannot guarantee unique track with maintain_track_cell_fraction  <0.5 ")

    # relates an individual feature family at a given time to a tracked feature family
    family_to_track_relationship = dict()

    # list of mergers/splits
    merge_split_list = list()

    # get all frame numbers we are dealing with 
    all_feat_frames = sorted(in_feat_arr['frame'].unique())

    # we don't need all variables here, just some of them. cut down to save computation time
    reduced_feat_arr = in_feat_arr[['frame', 'idx', 'feature', cell_column_name, feat_family_column_name]]
    for i, frame_num in enumerate(all_feat_frames):
        # pull the current time
        curr_time_feat_arr = reduced_feat_arr[reduced_feat_arr['frame']==frame_num]
        # get the relationship between feature family at this time -> cell at this time
        sets_curr_time = curr_time_feat_arr.groupby('feature_family_id')['cell'].agg(set)
        
        
        # first time we are dealing with 
        if i== 0: 
            # just set all feature families to the same tracked number
            sets_prev_time = copy.deepcopy(sets_curr_time)
            family_to_track_relationship = {sets_curr_time.index[tracked_num]: tracked_num+curr_tracked_family_num for tracked_num, _ in enumerate(sets_curr_time) }
            curr_tracked_family_num+=len(sets_curr_time)
            continue
            
            
        # from the perspective of the previous time, what is the intersection between cells at t=0 and t=1?
        family_intersections_through_time_prev = dict()
        # from the perspective of the current time, what is the intersection between cells at t=0 and t=1?
        family_intersections_through_time_curr = dict()
        # from the perspective of the 
        # loop through all families in the previous timestep
        for i_prev, prev_family_cells in enumerate(sets_prev_time):
            prev_family_label = sets_prev_time.index[i_prev]

            # loop through all families in the current timestep
            for i_curr, curr_family_cells in enumerate(sets_curr_time):
                curr_family_label = int(sets_curr_time.index[i_curr])

                # do the intersection of all cell numbers for a family at the previous timestep
                # with a family at the current timestep
                prev_curr_family_cell_overlap = prev_family_cells.intersection(curr_family_cells)
                # if we have any overlap, record it. 
                if len(prev_curr_family_cell_overlap) >0:
                    # record previous-> current links
                    if prev_family_label not in family_intersections_through_time_prev:
                        family_intersections_through_time_prev[prev_family_label] = [(curr_family_label, prev_curr_family_cell_overlap),]
                    else:
                        family_intersections_through_time_prev[prev_family_label].append((curr_family_label, prev_curr_family_cell_overlap))
                    
                    # record current ->previous links
                    if curr_family_label not in family_intersections_through_time_curr:
                        family_intersections_through_time_curr[curr_family_label] = [(prev_family_label, prev_curr_family_cell_overlap),]
                    else:
                        family_intersections_through_time_curr[curr_family_label].append((prev_family_label, prev_curr_family_cell_overlap))

                    
        
        # now that we have identified intersections, let's process them. 
        for i_curr, curr_family_cells in enumerate(sets_curr_time):
            curr_family_label = sets_curr_time.index[i_curr]
            if curr_family_label not in family_intersections_through_time_curr or len(family_intersections_through_time_curr[curr_family_label]) == 0:
                # we have no common cells between the current time and the previous time
                # give this family a new track number
                family_to_track_relationship[curr_family_label] = curr_tracked_family_num

                # increment current family track number
                curr_tracked_family_num+=1
                if curr_family_label in family_ids_of_interest:
                    print("Crashing out at no common cells")
                
            
            elif len(family_intersections_through_time_curr[curr_family_label]) == 1:
                # we have only one family that shares cells from the previous time with the current time
                prev_family_label = family_intersections_through_time_curr[curr_family_label][0][0]
                # check if the previous family can only be linked to this cell. 
                #print(prev_family_label, family_intersections_through_time_curr[curr_family_label])
                if len(family_intersections_through_time_prev[prev_family_label]) ==1:
                    family_to_track_relationship[curr_family_label] = family_to_track_relationship[prev_family_label]
                else:
                    # this family can only be linked to one previous cell, but the previous cell could be 
                    # linked to multiple current families (split)
                    common_cells_prev_curr = family_intersections_through_time_curr[curr_family_label][0][1]
                    all_prev_cells = sets_prev_time[prev_family_label]

                    # check if we can maintain family number based on our metric
                    keep_track_number = False
                    if maintain_family_metric == 'cells':
                        # check if our fraction of cells in the current time family is > our prescribed fraction
                        keep_track_number = len(common_cells_prev_curr)/len(all_prev_cells) > maintain_track_cell_fraction
                    elif maintain_family_metric =='area':
                        # check if our fraction of area maintained in the current family is > our prescribed fraction
                        curr_family_area = in_family_stat_arr.at[int(curr_family_label),'num_pixels']
                        prev_family_area = in_family_stat_arr.at[int(prev_family_label),'num_pixels']
                        keep_track_number = ((prev_family_area/curr_family_area) > maintain_track_area_fraction and
                                                (curr_family_area/prev_family_area) > maintain_track_area_fraction)
                        if keep_track_number:
                            pass
                            #print("Keeping track number. Location 1. prev_family: {0}, curr_family: {1}, prev_area: {2}, curr_area: {3}, prev_cells: ".format(
                            #    prev_family_label, curr_family_label, prev_family_area, curr_family_area
                            #))

                    else:
                        raise ValueError("Only acceptable metrics are 'cells' and 'area'.")

                    if keep_track_number:
                        # if it is, we can maintain the track number. 
                        family_to_track_relationship[curr_family_label] = family_to_track_relationship[prev_family_label]
                    else:
                        if curr_family_label in family_ids_of_interest:
                            print("Crashing out at too few cells")

                        # if it has too few cells from the previous family, it gets a new/unique number
                        family_to_track_relationship[curr_family_label] = curr_tracked_family_num
                        curr_tracked_family_num+=1
                        # log the split. 
                        merge_split_list.append((family_to_track_relationship[prev_family_label], family_to_track_relationship[curr_family_label] ))
                

            else:
                # >1 family that shares cells from the previous time with the current time
                prev_family_label = family_intersections_through_time_curr[curr_family_label][0][0]

                # check if this family's parents all only have one child
                children_family_ancestors = {cell_intersect_pair[0]: cell_intersect_pair[1]  for cell_intersect_pair in family_intersections_through_time_curr[curr_family_label]}
                max_parent_children = max([len(children_family_ancestors[x]) for x in children_family_ancestors])
                all_curr_cells = sets_curr_time[curr_family_label]
                if max_parent_children == 1:
                    # all parents only have this family as child cell. 
                    # many (prev) : one (curr) relationship
                    
                    found_parent_link = False
                    largest_prev_family_area = -1.0
                    largest_prev_family_id = -1
                    for parent_cell_label, parent_cell_intersection in family_intersections_through_time_curr[curr_family_label]:
                        
                        keep_track_number = False
                        if maintain_family_metric == 'cells':
                            # Link only if one parent has > maintain_track_cell_fraction of the child's cells
                            # this can only be triggered once
                            keep_track_number = (len(parent_cell_intersection)/len(all_curr_cells)) > maintain_track_cell_fraction
                            family_to_track_relationship[curr_family_label] = family_to_track_relationship[parent_cell_label]
                            found_parent_link = True
                            break

                        elif maintain_family_metric =='area':
                            # check if our fraction of area maintained in the current family is > our prescribed fraction
                            curr_family_area = in_family_stat_arr.at[int(curr_family_label),'num_pixels']
                            prev_family_area = in_family_stat_arr.at[int(parent_cell_label),'num_pixels']
                            keep_track_number = ((prev_family_area/curr_family_area) > maintain_track_area_fraction and
                                                    (curr_family_area/prev_family_area) > maintain_track_area_fraction)
                            
                            if keep_track_number:
                                if prev_family_area > largest_prev_family_area:
                                    largest_prev_family_area = prev_family_area
                                    largest_prev_family_id = family_to_track_relationship[parent_cell_label]

                        else:
                            raise ValueError("Only acceptable metrics are 'cells' and 'area'.")


                    # check that we are keeping largest family 
                    if maintain_family_metric == 'area' and largest_prev_family_id != -1:
                            family_to_track_relationship[curr_family_label] = largest_prev_family_id
                            found_parent_link = True

                    if not found_parent_link:
                        # all parents share less than maintain_track_cell of their child's cells. 
                        # issue this a new cell number.
                        if curr_family_label in family_ids_of_interest:
                            print("Crashing out at too small area")

                        family_to_track_relationship[curr_family_label] = curr_tracked_family_num
                        curr_tracked_family_num+=1
                    # record the merger, regardless.
                    for parent_cell_label, _ in family_intersections_through_time_curr[curr_family_label]:
                        merge_split_list.append((family_to_track_relationship[parent_cell_label], family_to_track_relationship[curr_family_label]))
                

                else:
                    # at least one parent has multiple children, meaning we have a many:many relationship. 
                    # We will only link if there is a parent cell where > maintain_track_cell_fraction of the cells in the child are there
                    # AND if the opposite fraction is also true
                    all_prev_cells = sets_prev_time[prev_family_label]
                    found_parent_link = False
                    largest_prev_family_area = -1.0
                    largest_prev_family_id = -1

                    for parent_cell_label, parent_cell_intersection in family_intersections_through_time_curr[curr_family_label]:

                        keep_track_number = False
                        if maintain_family_metric == 'cells':
                            # Link only if one parent has > maintain_track_cell_fraction of the child's cells
                            keep_track_number = ((len(parent_cell_intersection)/len(all_curr_cells)) > maintain_track_cell_fraction and 
                                                (len(parent_cell_intersection)/len(all_prev_cells)) > maintain_track_cell_fraction)
                            family_to_track_relationship[curr_family_label] = family_to_track_relationship[parent_cell_label]
                            found_parent_link = True

                        elif maintain_family_metric =='area':
                            # check if our fraction of area maintained in the current family is > our prescribed fraction
                                
                            curr_family_area = in_family_stat_arr.at[int(curr_family_label),'num_pixels']
                            prev_family_area = in_family_stat_arr.at[int(parent_cell_label),'num_pixels']
                            keep_track_number = ((prev_family_area/curr_family_area) > maintain_track_area_fraction and 
                                                (curr_family_area/prev_family_area) > maintain_track_area_fraction)
                            
                            if curr_family_label in family_ids_of_interest:
                                print(f"parent_cell_label: {parent_cell_label}, curr_family: {curr_family_label}, "
                                      f"prev_family_area: {prev_family_area}, curr_family_area: {curr_family_area},")


                            if keep_track_number:
                                if prev_family_area > largest_prev_family_area:
                                    largest_prev_family_area = prev_family_area
                                    largest_prev_family_id = family_to_track_relationship[parent_cell_label]


                        else:
                            raise ValueError("Only acceptable metrics are 'cells' and 'area'.")


                    # check that we are keeping largest family 
                    if maintain_family_metric == 'area' and largest_prev_family_id != -1:
                            family_to_track_relationship[curr_family_label] = largest_prev_family_id
                            found_parent_link = True
                    if not found_parent_link:
                        # all parents share less than maintain_track_cell of their child's cells. 
                        # issue this a new cell number.
                        if curr_family_label in family_ids_of_interest:
                            print("Crashing out at too few small area final", 
                                  "")

                        family_to_track_relationship[curr_family_label] = curr_tracked_family_num
                        curr_tracked_family_num+=1
                    
                    # record the merge/split mess
                    for parent_cell_label, _ in family_intersections_through_time_curr[curr_family_label]:
                        merge_split_list.append((family_to_track_relationship[parent_cell_label], family_to_track_relationship[curr_family_label]))

        # end of loop
        sets_prev_time = sets_curr_time

    # we have links between families and tracked families.
    # TODO: make into a frame again.
    out_df = copy.deepcopy(in_feat_arr) 
    tracked_family_arr = out_df[feat_family_column_name].apply(lambda x: family_to_track_relationship[x])
    out_df[tracked_family_column_name] = tracked_family_arr


    return out_df          

In [5]:
def vec_translate(array:np.array, my_dict: dict):    
    return np.vectorize(my_dict.__getitem__)(array)

In [6]:
def reassign_grid_coords_to_family(in_family_track_df: pd.DataFrame, in_grid:xr.DataArray, 
                                   family_id_name: str = 'feature_family_id', tracked_family_id_name: str = 'tracked_family_id'):
    """
    Function to renumber the input (i.e., per-frame) family ID to a per tracked family ID.

    Parameters
    ----------
    in_family_track_df: pd.DataFrame
        An output tracked dataframe from track_feature_families
    in_grid: xr.DataArray
        An output grid from family tracking
    family_id_name: str
        The name of the original untracked family ID column
    tracked_family_id_name: str
        The name of the tracked family ID column

    Returns 
    -------
    return_grid: xr.DataArray
        A grid with new renumbered values to reflect the tracked families
    """
    #feat_tracked_families.groupby(['frame','feature_family_id'])['tracked_family_id'].max().to_dict()
    
    #return_grid = copy.deepcopy(in_grid)

    dims_reordered = ['time']
    for curr_dim in in_grid.dims:
        if curr_dim != 'time':
            dims_reordered.append(curr_dim)
    return_grid = copy.deepcopy(in_grid.transpose(*dims_reordered))
    for i, (curr_time, grid_at_time) in enumerate(return_grid.groupby('time', squeeze=False)):
        curr_frame_seg = in_family_track_df[in_family_track_df['time'] == curr_time]
        
        map_family_to_track_id = {x :0 for x in np.unique(grid_at_time.values)}
        map_family_to_track_id.update(
                    dict(curr_frame_seg.groupby([family_id_name, tracked_family_id_name])['frame'].count().index.values))

        return_grid.values[i] = vec_translate(return_grid.values[i], map_family_to_track_id)
    
    return_grid = return_grid.rename("tracked_family_id")

    return return_grid

In [7]:
def combine_feature_families(in_feature_dfs: list[pd.DataFrame], in_stats: list[pd.DataFrame], in_grid: list[xr.DataArray] = None,
                             renumber_features: bool = False, old_feature_column_name=None, renumber_families: bool = True, 
                             old_family_column_name:str = "feature_family_id_original",
                             family_column_name: str = "feature_family_id",):
    """
    Function to combine dataframes of separately calculated feature families into one unified 
    dataframe for features.

    Parameters
    ----------
    in_feature_dfs: list[pd.DataFrame]
        List of dataframes generated by `identify_feature_families`
    in_stats: list[pd.DataFrame]
         List of statistics dataframes generated by `identify_feature_families`
    in_grid: list[xr.DataArray], optional
        List of DataArrays of the grids output by `identify_feature_families`. Warning that combining these is 
        computationally expensive.
    renumber_features: bool, optional (default: False)
        If true, features are renumber with contiguous integers. If false, the
        old feature numbers will be retained, but an exception will be raised if
        there are any non-unique feature numbers. If you have non-unique feature
        numbers and want to preserve them, use the old_feature_column_name to
        save the old feature numbers to under a different column name.
    old_feature_column_name: str or None, optional (default: None)
        The column name to preserve old feature numbers in. If None, these
        old numbers will be deleted. Users may want to enable this feature
        if they have run segmentation with the separate dataframes and
        therefore old feature numbers.
    renumber_families: bool, optional (default: True)
        If true, families are renumbered with contiguous integers. If false, the
        old family numbers will be retained, but an exception will be raised if
        there are any non-unique family numbers. If you have non-unique family
        numbers and want to preserve them, use the old_family_column_name to
        save the old family numbers to under a different column name.
    old_family_column_name: str or None, optional (default: "feature_family_id_original")
        The column name to preserve old family numbers in. If None, these
        old numbers will be deleted. Users may want to enable this feature
        if they have family identification with grid output enabled with the separate dataframes and
        therefore old family numbers.

    family_column_name: str
        The name in the output dataframe of the family ID

    Returns
    -------
    pd.DataFrame, pd.DataFrame, Optional xr.DataArray
        Combined feature dataframe, combined statistics dataframe, and combined grid if passed in. 

    """
    
    if in_grid is not None:
        raise NotImplementedError("Merging of grids is not yet supported")
    
    # combine the feature dataframes
    combined_feature_df = tobac.utils.general.combine_feature_dataframes(in_feature_dfs, renumber_features=renumber_features, old_feature_column_name=old_feature_column_name)
    #print(combined_feature_df)
    # get time: frame mapping
    time_frame_map = combined_feature_df.groupby('time')['frame'].max().to_dict()
    
    # now need to combine the stats dataframes
    combined_stats_df = pd.concat(in_stats)
    try:
        combined_stats_df["frame"] = [time_frame_map[x] for x in combined_stats_df['time']]
    except KeyError as e:
        return time_frame_map
    #print(combined_stats_df)


    if not renumber_families and np.any(
        np.bincount(combined_feature_df[family_column_name] + np.nanmin(combined_feature_df[family_column_name])) > 1
    ):
        raise ValueError(
            "Non-unique family values detected. Combining feature dataframes with original feature numbers"
            " cannot be performed because duplicate feature numbers exist, please use 'renumber_features=True'. "
            "If you would like to preserve the original feature numbers, please use the 'old_feature_column_name' "
            "keyword to define a new column for these values in the returned dataframe"
        )

    combined_stats_df = combined_stats_df.reset_index().set_index("feature_family_id", drop=False)

    if old_feature_column_name is not None:
        combined_feature_df[old_family_column_name] = copy.deepcopy(combined_feature_df[family_column_name])
        combined_stats_df[old_family_column_name] = copy.deepcopy(combined_stats_df[family_column_name])

    new_family_numbers = np.empty(len(combined_feature_df[family_column_name]), dtype=combined_feature_df[family_column_name].dtype)
    new_family_numbers_stats = np.empty(len(combined_stats_df[family_column_name]), dtype=combined_stats_df[family_column_name].dtype)
    family_number_map = dict()
    # let's start with the first frame's minimum number. 
    min_family_num = combined_feature_df[combined_feature_df['frame'] == combined_feature_df['frame'].min()][family_column_name].min()
    for i, (index, row) in enumerate(combined_feature_df.iterrows()):
        curr_fam_pair = (row['frame'], row[family_column_name])
        if curr_fam_pair in family_number_map:
            new_family_numbers[i] = family_number_map[curr_fam_pair]
        else:
            new_family_numbers[i] = min_family_num
            family_number_map[curr_fam_pair] = min_family_num
            min_family_num+=1
    
    # for i, (index, row) in enumerate(combined_stats_df.iterrows()):
    #     curr_fam_pair = (row['frame'], row[family_column_name])
    #     new_family_numbers_stats[i] = family_number_map[curr_fam_pair]

    for i, (index, row) in enumerate(combined_stats_df.iterrows()):
        curr_fam_pair = (row['frame'], row[family_column_name])
        if curr_fam_pair in family_number_map:
            new_family_numbers_stats[i] = family_number_map[curr_fam_pair]
        else:
            print(f"Warning: family {curr_fam_pair} in stats not found in feature map")
            new_family_numbers_stats[i] = -1  # or np.nan if preferred
    
    combined_feature_df[family_column_name] = new_family_numbers
    combined_stats_df[family_column_name] = new_family_numbers_stats
    
    combined_stats_df = combined_stats_df.reset_index(drop = True).set_index(family_column_name, drop=False)
    return combined_feature_df, combined_stats_df

##  Dask Client 

In [86]:
cluster = SLURMCluster(
    cores=4,  
    memory='20GB', 
    account='incus',
    walltime='94:00:00',
    scheduler_options={'dashboard_address': ':22015'},
    job_extra_directives=[
        '--partition=all',
        '--job-name=tobac',
        '--mail-type=BEGIN,END,FAIL',
        '--mail-user=rauth@colostate.edu'
    ]
)

client = Client(cluster)
cluster.scale(n=60)  # 60 workers × 4 cores = 240 cores total

In [90]:
client.close()
cluster.close()

## folders, times ,etc 

In [87]:
import datetime
aacp_base_folder = "/tempest/rauth/goes-data/combined_nc_dir_cleaned/"

out_base_folder = (
    "/tempest/rauth/DATA/tobac/masters_data/"
)

sector = 'M1'

version = 'v9.3.9_paper'

anvil_thresh = -20 # K (BTD)
dxy = 500 #m

start_time = datetime.datetime(2022, 3, 1)
end_time = datetime.datetime(2025, 8, 31)
break_time = datetime.timedelta(days = 1)
start_str = start_time.strftime("%Y_%m_%d")
end_str = end_time.strftime("%Y_%m_%d")


## ANVILS

### feature detection

In [88]:
def get_aacp_filenames(base_folder: str, curr_date: datetime.datetime):
    print("grabbing files")
    return glob.glob(base_folder + curr_date.strftime(f"%Y%m%d/OR_ABI_L1b_{sector}*.nc"))

def get_tobac_feats_aacp(aacp_file_name: str):
    data = xr.open_dataset(aacp_file_name)

    if 'tropopause_temperature' in data.data_vars:
        Features = tobac.feature_detection_multithreshold(
            (data.tropopause_temperature - data.ir_brightness_temperature), dxy, **parameters_features
        )
    else:
        return None
    data.close()
    return Features


def combine_tobac_feats(list_of_feats):
    print("combining features")
    if len(list_of_feats) == 0:
        return None
    return tobac.utils.general.combine_feature_dataframes(list_of_feats)

In [89]:
parameters_features = {
    'threshold': [0,-5, -10, -15, -20], #K, BTD
    'n_min_threshold': [1000, 2000, 4000, 4000, 4000], #m
    'position_threshold': 'weighted_diff',
    'sigma_threshold': 1,
    'min_distance': 12500, # m
    "target": "maximum",
    "PBC_flag": "none"
}
with open(out_base_folder + f'{start_str}_{end_str}_{version}_{sector}_anvil_features.json', 'w') as f:
    json.dump(parameters_features, f, indent=4)

all_break_times = pd.date_range(start_time, end_time, freq=break_time)
# Keep only March-August
all_break_times = all_break_times[all_break_times.month.isin([3,4,5,6,7,8])]
feature_list = []

# Register dask progress bar
pbar = ProgressBar()
pbar.register()

for curr_start_time in tqdm(all_break_times):
    print(curr_start_time)
    all_fnms = get_aacp_filenames(aacp_base_folder, curr_start_time)
    print(len(all_fnms))
    b = dask.bag.from_sequence(all_fnms, partition_size=1)
    print("making features")
    out_arr = dask.bag.map(lambda x: get_tobac_feats_aacp(x), b).compute()
    #out_arr.compute()
    out_arr = [x for x in out_arr if x is not None]
    # Skip processing if no valid data is present
    if not out_arr:
        print(f"No valid data for {curr_start_time}, skipping.")
        continue
    try:
        out_feats = combine_tobac_feats(out_arr)
        feature_list.append(out_feats)
    except ValueError as e:
        print(f"Error combining dataframes for chunk {curr_start_time}: {e}")
        continue


feature_df = tobac.utils.general.combine_feature_dataframes(feature_list)
print('features are combined')
feature_df.to_pickle(out_base_folder + f"/features/{version}_{sector}_anvil_features_{start_str}_{end_str}.p" )

  0%|          | 0/736 [00:00<?, ?it/s]

2022-03-01 00:00:00
grabbing files
1634
making features
combining features
2022-03-02 00:00:00
grabbing files
1739
making features
combining features
2022-03-03 00:00:00
grabbing files
1739
making features
combining features
2022-03-04 00:00:00
grabbing files
1437
making features
No valid data for 2022-03-04 00:00:00, skipping.
2022-03-05 00:00:00
grabbing files
1414
making features
combining features
2022-03-06 00:00:00
grabbing files
1433
making features
combining features
2022-03-07 00:00:00
grabbing files
1433
making features
combining features
2022-03-08 00:00:00
grabbing files
1435
making features
combining features
2022-03-09 00:00:00
grabbing files
1440
making features
combining features
2022-03-10 00:00:00
grabbing files
1440
making features
combining features
2022-03-11 00:00:00
grabbing files
1440
making features
combining features
2022-03-12 00:00:00
grabbing files
1440
making features
combining features
2022-03-13 00:00:00
grabbing files
1440
making features
combining feat

ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-66441288' coro=<Client._gather.<locals>.wait() done, defined at /home/rauth/miniforge3/envs/research/lib/python3.12/site-packages/distributed/client.py:2208> exception=AllExit()>
Traceback (most recent call last):
  File "/home/rauth/miniforge3/envs/research/lib/python3.12/site-packages/distributed/client.py", line 2217, in wait
    raise AllExit()
distributed.client.AllExit
ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-66441719' coro=<Client._gather.<locals>.wait() done, defined at /home/rauth/miniforge3/envs/research/lib/python3.12/site-packages/distributed/client.py:2208> exception=AllExit()>
Traceback (most recent call last):
  File "/home/rauth/miniforge3/envs/research/lib/python3.12/site-packages/distributed/client.py", line 2217, in wait
    raise AllExit()
distributed.client.AllExit
ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='

KeyboardInterrupt: 

ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-66441626' coro=<Client._gather.<locals>.wait() done, defined at /home/rauth/miniforge3/envs/research/lib/python3.12/site-packages/distributed/client.py:2208> exception=AllExit()>
Traceback (most recent call last):
  File "/home/rauth/miniforge3/envs/research/lib/python3.12/site-packages/distributed/client.py", line 2217, in wait
    raise AllExit()
distributed.client.AllExit
ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-66441689' coro=<Client._gather.<locals>.wait() done, defined at /home/rauth/miniforge3/envs/research/lib/python3.12/site-packages/distributed/client.py:2208> exception=AllExit()>
Traceback (most recent call last):
  File "/home/rauth/miniforge3/envs/research/lib/python3.12/site-packages/distributed/client.py", line 2217, in wait
    raise AllExit()
distributed.client.AllExit
ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='

### tracking

In [ ]:
parameters_linking = dict(
    method_linking  = "random",
    d_max           = 20000,    # m
    time_cell_min   = 1 * 60,   # s
    memory          = 2,
)
DT  = 60    # seconds
DXY = 500   # meters
Tracks_anvil= tobac.linking_trackpy(feature_df,None,dt=60,dxy=500,**parameters_linking)
Tracks_anvil.to_pickle(out_base_folder + f'/tracks/anvil_tracks_{version}_{sector}_{start_str}_{end_str}.p')

Frame 120420: 8 trajectories present.


In [59]:
Tracks_anvil_filled = fill_missing_timestamps(Tracks_anvil)
Tracks_anvil_filled.to_pickle(out_base_folder + f'/tracks/anvil_tracks_filled_{version}_{sector}_{start_str}_{end_str}.p')

with open(out_base_folder + f'{start_str}_{end_str}_{version}_{sector}_anvil_tracks.json', 'w') as f:
    json.dump(parameters_linking, f, indent=4)

### segmentation

In [60]:
def get_aacp_filenames(base_folder: str, curr_date: datetime.datetime):
    print("grabbing files for " + curr_date.strftime("%Y%m%d"))
    return glob.glob(base_folder + curr_date.strftime(f"%Y%m%d/OR_ABI_L1b_{sector}*.nc"))

def get_output_folder(base_folder: str, curr_date: datetime.datetime, is_anvil: bool):
    if is_anvil: 
        folder_check = base_folder + f"segments/anvils" + curr_date.strftime("/%Y%m%d/")
        os.makedirs(folder_check, exist_ok=True)
    else:
        folder_check = base_folder + f"segments/families" + curr_date.strftime("/%Y%m%d/")
        os.makedirs(folder_check, exist_ok=True)
    return folder_check


def get_tobac_seg_aacp(aacp_file_name: str):
    try:
        data = xr.open_dataset(aacp_file_name)
        print(f"Running on file: {aacp_file_name}, for time: {data.time.isel(time = 0).values}")
        
        #define the tracked features present at this time 
        goes_feats = pd.read_pickle(feature_file_name)
        goes_feat_dt = np.abs(goes_feats["time"] - data["time"].values[0])
        goes_feats_at_time = goes_feats[goes_feat_dt < datetime.timedelta(seconds=1)]
        del goes_feats
        
        # Check for features at this time
        if goes_feats_at_time.empty:
            print(f"No features found at time for {aacp_file_name}")
            data.close()
            return None
            
        # Filter for cells > 0
        valid_features = goes_feats_at_time[goes_feats_at_time['cell'] > 0]
        if valid_features.empty:
            print(f"No valid features (cell > 0) for {aacp_file_name}")
            data.close()
            return None
        
        parameters_segmentation = {}
        parameters_segmentation["method"] = "watershed"
        parameters_segmentation["threshold"] = anvil_thresh  
        parameters_segmentation["target"] = "maximum" 
        parameters_segmentation["PBC_flag"] = "none"
        dxy = 500

        if 'tropopause_temperature' not in data.data_vars:
            return None
            
        else:
            Mask_seg_anvil, Features_seg_anvil = tobac.segmentation_2D(
                valid_features, (data.tropopause_temperature - data.ir_brightness_temperature), 
                dxy, **parameters_segmentation
            )
        
            folder_name = get_output_folder(
                out_base_folder, pd.to_datetime(data["time"].values[0]), is_anvil=True
            )
            segments_anvil_file_name = pd.to_datetime(data["time"].values[0]).strftime(
                f"/{version}_single_seg_anvil_{anvil_thresh}_%Y_%m_%d_%H_%M_%S.nc"
            )
            
            Mask_seg_anvil.to_netcdf(
                folder_name + segments_anvil_file_name,
                engine="netcdf4",
                unlimited_dims=("time",),
            )
            seg_mask = Mask_seg_anvil.compute()
            
            # calculating statistics on segmentation 
            anvil_stats = {'min_tb_anvil': np.nanmin, 'mean_tb_anvil': np.nanmean}
            Features_seg_anvil = tobac.utils.bulk_statistics.get_statistics_from_mask(
                Features_seg_anvil, seg_mask, data.ir_brightness_temperature, statistic = anvil_stats
            )
        
            Features_seg_anvil.rename(columns={'ncells': 'ncells_anvil'}, inplace=True)

            # identify feature families 
            Feature_families, family_stats, Feature_families_mask = tobac.merge_split.families.identify_feature_families_from_segmentation(
                Features_seg_anvil[Features_seg_anvil['ncells_anvil'] > 0], Mask_seg_anvil, return_grid=True
            )
            family_stats["time"] = data["time"].values[0]
            
            segments_family_file_name = pd.to_datetime(data["time"].values[0]).strftime(
                f"/{version}_single_seg_family_{anvil_thresh}_%Y_%m_%d_%H_%M_%S.nc"
            )

            folder_name = get_output_folder(
                out_base_folder, pd.to_datetime(data["time"].values[0]), is_anvil=False
            )

            #saving family masks 
            Feature_families_mask.to_netcdf(
                folder_name + segments_family_file_name,
                engine="netcdf4",
                unlimited_dims=("time",),
            )
            
            print("Returning families for:", aacp_file_name)
            data.close()
        
        # Validate results before returning
        if isinstance(Feature_families, pd.DataFrame) and isinstance(family_stats, pd.DataFrame):
            if not Feature_families.empty and not family_stats.empty:
                print(f"Successfully processed {aacp_file_name}: {Feature_families.shape[0]} features, {family_stats.shape[0]} stats")
                return Feature_families, family_stats
            else:
                print(f"Empty results for {aacp_file_name}")
                return None
        else:
            print(f"Invalid result types for {aacp_file_name}")
            return None
            
    except Exception as e:
        print(f"Failed to process {aacp_file_name}: {e}")
        print(f"Exception type: {type(e)}")
        import traceback
        traceback.print_exc()
        return None

def combine_tobac_families(list_of_families, list_of_stats):
    if len(list_of_families) == 0:
        return None
    return combine_feature_families(list_of_families, list_of_stats, renumber_features=True, old_feature_column_name='feature_anvil_id') #feature_mask_seg


In [61]:
feature_file_name = (
    out_base_folder + f"tracks/anvil_tracks_filled_{version}_{sector}_{start_str}_{end_str}.p"
)

break_time = datetime.timedelta(days = 1)
break_day_into_n_chunks = 24

#original
feature_list = []
stat_list = []

# Date setup
all_break_times = pd.date_range(start_time, end_time, freq=break_time)
all_break_times = all_break_times[all_break_times.month.isin([3,4,5,6,7,8])]

# Collect all tasks
all_bags = []

for curr_start_time in tqdm(all_break_times):
    all_fnms_in_day = get_aacp_filenames(aacp_base_folder, curr_start_time)

    for i in range(break_day_into_n_chunks):
        start_val = i * (len(all_fnms_in_day) // break_day_into_n_chunks)
        end_val = min(len(all_fnms_in_day), (i + 1) * (len(all_fnms_in_day) // break_day_into_n_chunks))

        print(f"Registering chunk {start_val} to {end_val}")
        all_fnms = all_fnms_in_day[start_val:end_val]

        if not all_fnms:
            continue

        b = dask.bag.from_sequence(all_fnms, partition_size=12).map(get_tobac_seg_aacp)
        all_bags.append(b)

# Combine all Dask bags into one
if not all_bags:
    print("No tasks created.")
    exit()

combined_bag = dask.bag.concat(all_bags)

print("Starting computation...")
with Profiler() as prof, ResourceProfiler() as rprof, CacheProfiler() as cprof, ProgressBar():
    results = combined_bag.compute()
print("Computation complete!")

# Process results
cleaned_results = [r for r in results if isinstance(r, tuple) and len(r) == 2 and all(isinstance(x, pd.DataFrame) for x in r)]

if not cleaned_results:
    print("No valid data found in any chunk.")
    exit()

# Group and combine
for idx, r in enumerate(cleaned_results):
    df1, df2 = r
    if df1.empty or df2.empty:
        continue
    try:
        out_feats, out_stats = df1, df2 
        feature_list.append(out_feats)
        stat_list.append(out_stats)
        out_feats.to_pickle(out_base_folder + f"/features/families/{version}_{sector}_feat_seg_anvil{anvil_thresh}_anvil_families_chunk{idx}.p")
        out_stats.to_pickle(out_base_folder + f"/features/family_stats/{version}_{sector}_feat_seg_anvil{anvil_thresh}_anvil_family_stats_chunk{idx}.p")
    except ValueError as e:
        print(f"Error combining dataframes in chunk {idx}: {e}")
        continue
print('finished computation!')

# Final combination across all valid chunks
if feature_list:
    print("Combining final families...")
    feat_seg_anvil_df, family_stat_df = combine_feature_families(feature_list, stat_list, renumber_features=True, old_feature_column_name='feature_anvil_id2')
    feat_seg_anvil_df.to_pickle(out_base_folder + f"/features/{version}_{sector}_feat_seg_anvil{anvil_thresh}_anvil_families_{start_str}_{end_str}.p")
    family_stat_df.to_pickle(out_base_folder + f"/features/{version}_{sector}_feat_seg_anvil{anvil_thresh}_anvil_family_stats_{start_str}_{end_str}.p")
    print('families are combined!')
else:
    print("No features to combine across all time chunks.")

  0%|          | 0/184 [00:00<?, ?it/s]

grabbing files for 20250301
Registering chunk 0 to 59
Registering chunk 59 to 118
Registering chunk 118 to 177
Registering chunk 177 to 236
Registering chunk 236 to 295
Registering chunk 295 to 354
Registering chunk 354 to 413
Registering chunk 413 to 472
Registering chunk 472 to 531
Registering chunk 531 to 590
Registering chunk 590 to 649
Registering chunk 649 to 708
Registering chunk 708 to 767
Registering chunk 767 to 826
Registering chunk 826 to 885
Registering chunk 885 to 944
Registering chunk 944 to 1003
Registering chunk 1003 to 1062
Registering chunk 1062 to 1121
Registering chunk 1121 to 1180
Registering chunk 1180 to 1239
Registering chunk 1239 to 1298
Registering chunk 1298 to 1357
Registering chunk 1357 to 1416
grabbing files for 20250302
Registering chunk 0 to 60
Registering chunk 60 to 120
Registering chunk 120 to 180
Registering chunk 180 to 240
Registering chunk 240 to 300
Registering chunk 300 to 360
Registering chunk 360 to 420
Registering chunk 420 to 480
Registeri

/home/rauth/miniforge3/envs/research/lib/python3.12/site-packages/distributed/client.py:3162: UserWarning: Sending large graph of size 36.74 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


Computation complete!
finished computation!
Combining final families...
families are combined!


### tracked families

In [62]:
Tracked_feature_families = track_feature_families(feat_seg_anvil_df, family_stat_df.set_index('feature_family_id'))
Tracked_feature_families.to_pickle(out_base_folder + f"/tracks/tracked_families_{version}_{sector}_{anvil_thresh}_{start_str}_{end_str}.p")

In [63]:
# saving tracked family masks 
data_dates = pd.date_range(
    pd.Timestamp('2022-03-01 00:00:00'),
    pd.Timestamp('2025-08-31 23:59:00'),
    freq='1d'
).strftime("%Y%m%d")
 
file_glob = f'{version}_single_seg_family*'
data_dir = Path("/tempest/rauth/DATA/tobac/masters_data/segments/families")

def filenames_from_date(dt_str):
    return [str(x) for x in data_dir.joinpath(dt_str).glob(file_glob)]

def get_tracked_df():
    worker = get_worker()
    if not hasattr(worker, "tracked_df"):
        # Load once
        full_df = pd.read_pickle(f'/tempest/rauth/DATA/tobac/masters_data/tracks/tracked_families_{version}_{sector}_{anvil_thresh}_{start_str}_{end_str}.p')
        # Only keep necessary columns
        worker.tracked_df = full_df[['time', 'feature_family_id_original', 'tracked_family_id', 'frame']]
    return worker.tracked_df

output_base_dir = Path("/tempest/rauth/DATA/tobac/masters_data/segments/families/tracked")

@dask.delayed
def process_mask_batch(file_list):
    results = []
    tracked_families = get_tracked_df()
    
    for this_file in file_list:
        try:
            ds = xr.open_dataset(this_file)
            
            tracked = reassign_grid_coords_to_family(
                tracked_families,
                ds.family_grid,
                family_id_name='feature_family_id_original',
                tracked_family_id_name='tracked_family_id'
            )
            
            ds['tracked_family_mask'] = tracked
            
            # Extract date from path and construct new output path
            # Assuming path is like .../families/YYYYMMDD/filename.nc
            date_str = Path(this_file).parent.name  # Gets YYYYMMDD
            filename = Path(this_file).name
            
            # Create output directory if needed
            output_dir = output_base_dir / date_str
            output_dir.mkdir(parents=True, exist_ok=True)
            
            # Save in tracked directory
            output_path = output_dir / filename.replace(".nc", "_tracked.nc")
            
            ds.to_netcdf(
                str(output_path), 
                engine="netcdf4", 
                unlimited_dims=("time",),
                encoding={
                    'tracked_family_mask': {'zlib': True, 'complevel': 1}
                }
            )
            ds.close()
            results.append(str(output_path))
            
        except Exception as e:
            print(f"❌ Failed on {this_file}: {e}")
            results.append(None)
    
    return results

all_filenames = [x for d in data_dates for x in filenames_from_date(d)]
all_filenames = [f for f in all_filenames if f'{version}' in f]

chunk_size = 200
concurrent_batches = 60

chunks = [
    all_filenames[i:i + chunk_size]
    for i in range(0, len(all_filenames), chunk_size)
]

with tqdm(total=len(all_filenames), desc="Files processed") as pbar:
    for i in range(0, len(chunks), concurrent_batches):
        batch_chunks = chunks[i:i + concurrent_batches]

        tasks = [
            process_mask_batch(chunk)
            for chunk in batch_chunks
        ]

        results = dask.compute(*tasks)

        # Count completed files
        n_done = sum(len(r) for r in results if r is not None)
        pbar.update(n_done)

Files processed:   0%|          | 0/215449 [00:00<?, ?it/s]

In [64]:
def fill_interpolated_values(df, column1='ncells_anvil', column2='min_tb_anvil'):
    # Helper: fill column if both neighbors are valid
    def interpolate_if_valid(col):
        mask = (df[col] == 0) | (df[col].isna())
        fwd = df.groupby('cell')[col].shift(-1)
        bwd = df.groupby('cell')[col].shift(1)
        # Only fill where both neighbors exist and are >0 / not NaN
        valid_neighbors = (~fwd.isna()) & (~bwd.isna()) & (fwd != 0) & (bwd != 0)
        fill_mask = mask & valid_neighbors
        df.loc[fill_mask, col] = ((fwd + bwd) / 2)[fill_mask]

    # Apply to numeric columns
    interpolate_if_valid(column1)
    interpolate_if_valid(column2)

    return df
    
Tracked_feature_families_drop = Tracked_feature_families.loc[:, ~Tracked_feature_families.columns.duplicated()]
Tracked_feature_families_drop = fill_interpolated_values(Tracked_feature_families_drop)

In [65]:
# Calculate family-level statistics
Tracked_feature_families_drop['weighted_tb'] = (
    Tracked_feature_families_drop['mean_tb_anvil'] * 
    Tracked_feature_families_drop['ncells_anvil']
)

family_stats = Tracked_feature_families_drop.groupby(["tracked_family_id", "time"]).agg({
    'ncells_anvil': 'sum',           # Total area
    'weighted_tb': 'sum',             # Sum of weighted temperatures
    'min_tb_anvil': 'min'             # Coldest pixel across all anvils
}).reset_index()

# Calculate area-weighted mean TB
family_stats['mean_tb_family'] = family_stats['weighted_tb'] / family_stats['ncells_anvil']

# Rename and clean up
family_stats = family_stats.rename(columns={
    "ncells_anvil": "ncells_family",
    "min_tb_anvil": "min_tb_family"
}).drop(columns=['weighted_tb'])

# Merge family-level stats back to original dataframe
final_families = Tracked_feature_families_drop.merge(
    family_stats, 
    how='left', 
    on=['tracked_family_id', 'time']
)

# Save
final_families.to_pickle(
    out_base_folder + f"/features/final_families_{version}_{sector}_{anvil_thresh}_{start_str}_{end_str}.p"
)

print(f"✓ Saved {len(final_families)} rows with family-level statistics")
print(f"  Columns added: ncells_family, mean_tb_family, min_tb_family")

✓ Saved 1103800 rows with family-level statistics
  Columns added: ncells_family, mean_tb_family, min_tb_family


## OTs

### feature detection

In [45]:
def get_aacp_filenames(base_folder: str, curr_date: datetime.datetime):
    print("grabbing files")
    return glob.glob(base_folder + curr_date.strftime(f"%Y%m%d/OR_ABI_L1b_{sector}*.nc"))

def get_tobac_feats_aacp(aacp_file_name: str):
    try: 
        data = xr.open_dataset(aacp_file_name)
        parameters_features = {}
        parameters_features["position_threshold"] = "extreme"
        parameters_features["sigma_threshold"] = .5
        parameters_features["n_erosion_threshold"] = 0
        
        parameters_features["n_min_threshold"] = 0 
        parameters_features["min_distance"] = 10000 # meters 
        parameters_features["target"] = "maximum"
        parameters_features["PBC_flag"] = "none"
        dxy = 500
    
        if 'vis_tropdiff_ot' in data.data_vars:
            parameters_features["threshold"] = 0.1
            Features = tobac.feature_detection_multithreshold(
                data.vis_tropdiff_ot, dxy, **parameters_features
            )
        elif 'tropdifflin_ot' in data.data_vars:
            parameters_features["threshold"] = 0.4
            Features = tobac.feature_detection_multithreshold(
                data.tropdifflin_ot, dxy, **parameters_features
            )
        else:
            print("no valid variables found!")
        
        data.close()
        with open(out_base_folder + f'{start_str}_{end_str}_{version}_{sector}_features.json', 'w') as f:
            json.dump(parameters_features, f, indent=4)
        return Features
    except Exception as e:
        print(f"❌ Error on {aacp_file_name}: {e}")
    return None


def combine_tobac_feats(list_of_feats):
    print("combining features")
    if len(list_of_feats) == 0:
        return None
    return tobac.utils.general.combine_feature_dataframes(list_of_feats)

In [47]:
all_break_times = pd.date_range(start_time, end_time, freq=break_time)
all_break_times = all_break_times[all_break_times.month.isin([3,4,5,6,7,8])]
feature_list = []

# Register dask progress bar
pbar = ProgressBar()
pbar.register()

for curr_start_time in tqdm(all_break_times):
    print(curr_start_time)
    all_fnms = get_aacp_filenames(aacp_base_folder, curr_start_time)
    print(len(all_fnms))
    b = dask.bag.from_sequence(all_fnms, partition_size=9)
    print("making features")
    out_arr = dask.bag.map(lambda x: get_tobac_feats_aacp(x), b).compute()
    out_arr = [x for x in out_arr if x is not None]
    # Skip processing if no valid data is present
    if not out_arr:
        print(f"No valid data for {curr_start_time}, skipping.")
        continue
    try:
        out_feats = combine_tobac_feats(out_arr)
        curdatstrYmd = curr_start_time.strftime("%Y_%m_%d_%H.p")
        feature_list.append(out_feats)
    except ValueError as e:
        print(f"Error combining dataframes for chunk {curr_start_time}: {e}")
        continue


feature_df = tobac.utils.general.combine_feature_dataframes(feature_list)
feature_df.to_pickle(out_base_folder + f"/features/{version}_{sector}_ot_features_{start_str}_{end_str}.p" )

  0%|          | 0/736 [00:00<?, ?it/s]

2022-03-01 00:00:00
grabbing files
1634
making features
combining features
2022-03-02 00:00:00
grabbing files
1739
making features
combining features
2022-03-03 00:00:00
grabbing files
1739
making features
combining features
2022-03-04 00:00:00
grabbing files
1437
making features
combining features
2022-03-05 00:00:00
grabbing files
1414
making features
combining features
2022-03-06 00:00:00
grabbing files
1433
making features
combining features
2022-03-07 00:00:00
grabbing files
1433
making features
combining features
2022-03-08 00:00:00
grabbing files
1435
making features
combining features
2022-03-09 00:00:00
grabbing files
1440
making features
combining features
2022-03-10 00:00:00
grabbing files
1440
making features
combining features
2022-03-11 00:00:00
grabbing files
1440
making features
combining features
2022-03-12 00:00:00
grabbing files
1440
making features
combining features
2022-03-13 00:00:00
grabbing files
1440
making features
No valid data for 2022-03-13 00:00:00, skipp

### tracking

In [10]:
parameters_linking = {}
parameters_linking["method_linking"] = "predict"
parameters_linking["adaptive_stop"] = 0.2
parameters_linking["adaptive_step"] = 0.95
parameters_linking["extrapolate"] = 0
parameters_linking["order"] = 1
parameters_linking["subnetwork_size"] = 8
parameters_linking["memory"] = 2
parameters_linking["v_max"] = 125 #m/s
parameters_linking["time_cell_min"] = 2 * 60  # 2 minute minimum cell time
print("start tracking")
Tracks_ot= tobac.linking_trackpy(feature_df,None,dt=60,dxy=500,**parameters_linking)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [16]:
Tracks_ot_filled = fill_missing_timestamps(Tracks_ot)

In [ ]:
Tracks_ot_filled.to_pickle(out_base_folder + f'/tracks/ot_tracks_{version}_{sector}_{start_str}_{end_str}.p')
with open(out_base_folder + f'{start_str}_{end_time}_{version}_{sector}_ot_tracks.json', 'w') as f:
    json.dump(parameters_linking, f, indent=4)

### segmentation

In [60]:
def get_aacp_filenames(base_folder: str, curr_date: datetime.datetime):
    print("grabbing files for " + curr_date.strftime("%Y%m%d"))
    return glob.glob(base_folder + curr_date.strftime(f"%Y%m%d/OR_ABI_L1b_{sector}*.nc"))

def get_output_folder(base_folder: str, curr_date: datetime.datetime):
    folder_check = base_folder + f"segments/ots" + curr_date.strftime("/%Y%m%d/")
    os.makedirs(folder_check, exist_ok=True)
    return folder_check

def get_tobac_seg_aacp(aacp_file_name: str):
    try:
        data = xr.open_dataset(aacp_file_name)
        print('opened', aacp_file_name)
        # Filter features for this timestamp
        goes_feats = pd.read_pickle(feature_file_name)
        goes_feat_dt = np.abs(goes_feats["time"] - data["time"].values[0])
        goes_feats_at_time = goes_feats[goes_feat_dt < datetime.timedelta(seconds=1)]
        del goes_feats

        dxy = 500
        # Try OT segmentation
        if "vis_tropdiff_ot" in data.data_vars:
            data_ot = data.vis_tropdiff_ot.compute()
            thresh_val = 0.1
        elif "tropdifflin_ot" in data.data_vars:
            data_ot = data.tropdifflin_ot.compute()
            thresh_val = 0.4
        else:
            print(f"⚠️ Skipping {aacp_file_name}: no valid variables found.")
            return None

        parameters_segmentation_ot = {
            "method": "watershed",
            "target": "maximum",
            "PBC_flag": "none",
            'threshold': thresh_val
        }

        print('running segmentation')
        goes_feats_thresh = goes_feats_at_time[goes_feats_at_time['threshold_value'] == thresh_val]

        if goes_feats_thresh.empty:
            print(f"⚠️ No features at threshold {thresh_val} for {aacp_file_name}, skipping.")
            return None
        
        Mask_seg_ot, Features_seg_ot = tobac.segmentation_2D(
            goes_feats_thresh,
            data_ot, dxy, **parameters_segmentation_ot
        )

        #calculate statistics on OT segments 
        seg_mask = Mask_seg_ot.compute()
        Features_seg_ot = tobac.utils.bulk_statistics.get_statistics_from_mask(
            Features_seg_ot, seg_mask, data.ir_brightness_temperature, statistic={'min_tb_ot': np.nanmin}
        )

        if 'tropopause_temperature' in data.data_vars:
            Features_seg_ot = tobac.utils.bulk_statistics.get_statistics_from_mask(
                Features_seg_ot, seg_mask, data.tropopause_temperature, statistic={'trop_temp': np.nanmean}
            )
        else:
            Features_seg_ot == Features_seg_ot
        
        folder_name = get_output_folder(
            out_base_folder, pd.to_datetime(data["time"].values[0])
        )
        segments_ot_file_name = pd.to_datetime(data["time"].values[0]).strftime(
            f"/{version}_single_seg_ot_{anvil_thresh}_%Y_%m_%d_%H_%M_%S.nc"
        )
    
        Mask_seg_ot.to_netcdf(
            folder_name + segments_ot_file_name,
            engine="netcdf4",
            unlimited_dims=("time",),
        )

        Features_seg_ot.rename(columns={'ncells': 'ncells_ot'}, inplace=True)
        return Features_seg_ot

    except Exception as e:
        print(f"❌ Error processing {aacp_file_name}: {e}")
        traceback.print_exc()  # shows exactly which line/library triggered an error
        return None

    finally:
        try:
            print(f"Successfully processed {aacp_file_name}")
            data.close()
        except:
            pass


In [61]:
feature_file_name = (
    out_base_folder + f"tracks/ot_tracks_{version}_{sector}_{start_str}_{end_str}.p"
)
break_time = datetime.timedelta(days = 1)
break_day_into_n_chunks = 24

feature_list = []

# Register dask progress bar
pbar = ProgressBar()
pbar.register()

all_break_times = pd.date_range(start_time, end_time, freq=break_time)
all_break_times = all_break_times[all_break_times.month.isin([3,4,5,6,7,8])]
curdatstrYm = start_time.strftime("%Y_%m.p")
curdatstrYmd = start_time.strftime("%Y_%m_%d.p")

for curr_start_time in tqdm(all_break_times):

    all_fnms_in_day = get_aacp_filenames(aacp_base_folder, curr_start_time)

    for i in range(break_day_into_n_chunks):
        start_val = i * (len(all_fnms_in_day) // (break_day_into_n_chunks))
        end_val = min(
            (
                len(all_fnms_in_day),
                (i + 1) * (len(all_fnms_in_day) // (break_day_into_n_chunks)),
            )
        )
        print("Working from {0} to {1}".format(start_val, end_val))
        all_fnms = all_fnms_in_day[start_val:end_val]
        b = dask.bag.from_sequence(all_fnms, partition_size=1)
        out_arr = b.map(lambda x: get_tobac_seg_aacp(x)).compute()
        # Filter out None values from results
        out_arr = [x for x in out_arr if x is not None]
        # Skip processing if no valid data is present
        if not out_arr:
            print(f"No valid data for chunk {start_val}-{end_val}, skipping.")
            continue
        
        try:
            out_feats = tobac.utils.general.combine_feature_dataframes(
                out_arr, old_feature_column_name='feature_mask_seg'
            )
            feature_list.append(out_feats)
        except ValueError as e:
            print(f"Error combining dataframes for chunk {start_val}-{end_val}: {e}")
            continue

# Combine all valid features into a final DataFrame
if feature_list:
    feat_seg_ot_df = tobac.utils.general.combine_feature_dataframes(feature_list)
    feat_seg_ot_df.to_pickle(out_base_folder + f"/features/{version}_{sector}_feat_seg_ot{anvil_thresh}_{start_str}_{end_str}.p" )
else:
    print("No features to combine across all time chunks.")


  0%|          | 0/736 [00:00<?, ?it/s]

grabbing files for 20220301
Working from 0 to 68
Working from 68 to 136
Working from 136 to 204
No valid data for chunk 136-204, skipping.
Working from 204 to 272
No valid data for chunk 204-272, skipping.
Working from 272 to 340
Working from 340 to 408
No valid data for chunk 340-408, skipping.
Working from 408 to 476
No valid data for chunk 408-476, skipping.
Working from 476 to 544
No valid data for chunk 476-544, skipping.
Working from 544 to 612
Working from 612 to 680
No valid data for chunk 612-680, skipping.
Working from 680 to 748
Working from 748 to 816
No valid data for chunk 748-816, skipping.
Working from 816 to 884
No valid data for chunk 816-884, skipping.
Working from 884 to 952
No valid data for chunk 884-952, skipping.
Working from 952 to 1020
Working from 1020 to 1088
No valid data for chunk 1020-1088, skipping.
Working from 1088 to 1156
Working from 1156 to 1224
No valid data for chunk 1156-1224, skipping.
Working from 1224 to 1292
No valid data for chunk 1224-1292,

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



Working from 900 to 960
Working from 960 to 1020
Working from 1020 to 1080
Working from 1080 to 1140
Working from 1140 to 1200
Working from 1200 to 1260
Working from 1260 to 1320
Working from 1320 to 1380
Working from 1380 to 1440
grabbing files for 20240515
Working from 0 to 60
Working from 60 to 120
Working from 120 to 180
Working from 180 to 240
Working from 240 to 300
Working from 300 to 360
Working from 360 to 420
Working from 420 to 480
Working from 480 to 540
Working from 540 to 600
Working from 600 to 660
Working from 660 to 720
Working from 720 to 780
Working from 780 to 840
Working from 840 to 900
Working from 900 to 960
Working from 960 to 1020
Working from 1020 to 1080
Working from 1080 to 1140
Working from 1140 to 1200
Working from 1200 to 1260
Working from 1260 to 1320
Working from 1320 to 1380
Working from 1380 to 1440
grabbing files for 20240516
Working from 0 to 60
Working from 60 to 120
Working from 120 to 180
Working from 180 to 240
Working from 240 to 300
Working fr

In [42]:
def fill_interpolated_values(df, column1='ncells_ot', column2 ='min_tb_ot'):
    # Identify rows where the value is 0 and likely interpolated
    mask1 = (df[column1] == 0) | (df[column1].isna()) 

    # Forward and backward fill within each cell group
    forward1 = df.groupby('cell')[column1].shift(-1)
    backward1 = df.groupby('cell')[column1].shift(1)

    # Fill only interpolated values with average of neighbor values
    df.loc[mask1, column1] = ((forward1 + backward1) / 2)[mask1]

    # Identify rows where the value is 0 and likely interpolated
    mask2 = (df[column2] == 0) | (df[column2].isna()) 

    # Forward and backward fill within each cell group
    forward2 = df.groupby('cell')[column2].shift(-1)
    backward2 = df.groupby('cell')[column2].shift(1)

    # Fill only interpolated values with average of neighbor values
    df.loc[mask2, column2] = ((forward2 + backward2) / 2)[mask2]
    return df

final_ots = fill_interpolated_values(feat_seg_ot_df)

## MATCHING

In [68]:
def load_mask_fn(t):
    dt = pd.to_datetime(str(t))
    day_dir = dt.strftime("%Y%m%d")
    timestamp_str = dt.strftime("%Y_%m_%d_%H_%M_%S")
    base_dir = "DATA/tobac/masters_data/segments/families/tracked/"
    file_path = os.path.join(base_dir, day_dir, f"{version}_single_seg_family_-20_{timestamp_str}_tracked.nc")
    print(f"🔍 Loading mask: {file_path}")
    if not os.path.exists(file_path):
        print(f"⚠️ File not found: {file_path}")
        raise FileNotFoundError(file_path)
    ds = xr.open_dataset(file_path)
    return ds["family_grid"], ds["tracked_family_mask"]

def match_ots_to_anvil_chunk_batched(df_chunk, load_mask_fn):
    print(f"⚡ match_ots_to_anvil_chunk_batched called with {len(df_chunk)} rows")
    df_chunk = df_chunk.copy()
    
    # Initialize both columns
    df_chunk["family_mask_id"] = -1
    df_chunk["tracked_family_id"] = -1
    
    # Round timestamps to nearest second for grouping (or use floor)
    df_chunk["time_rounded"] = pd.to_datetime(df_chunk["time"]).dt.floor("S")
    grouped = df_chunk.groupby("time_rounded")
    
    for t_round, group in grouped:
        try:
            # Load both masks
            seg_mask, tracked_mask = load_mask_fn(t_round)
            
            lat_grid = seg_mask.latitude.values
            lon_grid = seg_mask.longitude.values
            
            # Get the values for both masks
            seg_grid = seg_mask.squeeze().values
            tracked_grid = tracked_mask.squeeze().values
            
            if lat_grid.ndim == 1 and lon_grid.ndim == 1:
                lon_grid, lat_grid = np.meshgrid(lon_grid, lat_grid)
            
            for idx, row in group.iterrows():
                lat_ot = row["latitude"]
                lon_ot = row["longitude"]
                
                # Find nearest grid point
                dist2 = (lat_grid - lat_ot)**2 + (lon_grid - lon_ot)**2
                iy, ix = np.unravel_index(np.argmin(dist2), dist2.shape)
                
                # Extract values from both masks at the same location
                seg_feature_id = int(seg_grid[iy, ix])
                tracked_feature_id = int(tracked_grid[iy, ix])
                
                # Store both values
                df_chunk.at[idx, "family_mask_id"] = seg_feature_id
                df_chunk.at[idx, "tracked_family_id"] = tracked_feature_id
                
        except Exception as e:
            print(f"⚠️ Failed at time {t_round}: {e}")
    
    return df_chunk.drop(columns="time_rounded")

def match_chunk(chunk):
    print(f"🚀 Starting chunk with {len(chunk)} rows")
    return match_ots_to_anvil_chunk_batched(chunk, load_mask_fn)

# Chunking
n_chunks = max(10, len(final_ots) // 5000)
df_chunks = np.array_split(final_ots, n_chunks)

# Submit real futures to the Dask cluster
futures = [client.submit(match_chunk, chunk) for chunk in df_chunks]

results = []
with tqdm(total=n_chunks, desc="Chunks processed") as pbar:
    for future in as_completed(futures):
        results.append(future.result())
        pbar.update(1)

/home/rauth/miniforge3/envs/research/lib/python3.12/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Chunks processed:   0%|          | 0/1339 [00:00<?, ?it/s]

In [69]:
#Combine results
matched_ots = pd.concat(results, ignore_index=True)
matched_ots = matched_ots.sort_values(["time", "cell"]).reset_index(drop=True)
matched_ots = matched_ots[matched_ots['time'].dt.month.isin(range(3, 9))]
matched_ots.to_pickle(out_base_folder + f"/features/final_ots_matched_{version}_{sector}_-20_{start_str}_{end_str}.p")


## QC

In [77]:
final_anvils = final_families.copy()
final_ots = matched_ots.copy()

In [78]:
# calculate time_family
first_time_per_family = final_anvils.groupby('tracked_family_id')['time'].min()
final_anvils['time_family'] = final_anvils['time'] - final_anvils['tracked_family_id'].map(first_time_per_family)

In [79]:
print(f"initial OTs rows:    {len(final_ots):,}")
print(f"initial Anvil rows:  {len(final_anvils):,}\n")

def clean_family_ids_temporal(df, cell_col="cell", tf_col="tracked_family_id"):
    cleaned = []
    for cid, g in df.groupby(cell_col):
        g = g.sort_values("time")  
        fam_orig = g[tf_col].to_numpy()

        # If cell has no valid IDs at all, skip it
        if not (fam_orig > 0).any():
            continue

        fam_clean = fam_orig.copy()
        n = len(fam_clean)

        for i in range(n):
            # Skip already valid IDs
            if fam_clean[i] > 0:
                continue

            # Look backward for previous valid ID
            prev_id = fam_clean[i-1] if i > 0 and fam_clean[i-1] > 0 else None
            # Look forward for next valid ID
            next_id = fam_clean[i+1] if i < n-1 and fam_clean[i+1] > 0 else None

            # Case 1: both neighbors valid and equal
            if prev_id and next_id and prev_id == next_id:
                fam_clean[i] = prev_id
            # Case 2: only previous valid
            elif prev_id and not next_id:
                fam_clean[i] = prev_id
            # Case 3: only next valid
            elif next_id and not prev_id:
                fam_clean[i] = next_id
            # Case 4: neighbors disagree → use local majority in ±2 window
            elif prev_id and next_id and prev_id != next_id:
                window = fam_clean[max(0, i-2): min(n, i+3)]
                local_valid = window[window > 0]
                if len(local_valid) > 0:
                    fam_clean[i] = pd.Series(local_valid).mode().iloc[0]
            # Case 5: still invalid → fill with nearest valid ID in the whole cell
            elif (prev_id is None and next_id is None):
                # Use nearest valid ID anywhere in the cell
                valid_indices = np.where(fam_clean > 0)[0]
                if len(valid_indices) > 0:
                    nearest_idx = valid_indices[np.abs(valid_indices - i).argmin()]
                    fam_clean[i] = fam_clean[nearest_idx]
                
        # Store cleaned values in a new column
        g = g.copy()
        g["tracked_family_id_clean"] = fam_clean
        cleaned.append(g)

    # Concatenate all cells back
    return pd.concat(cleaned)


def fill_interpolated_values_ot(df, column1='ncells_ot', column2='min_tb_ot'):
    df = df.copy()
    for col in [column1, column2]:

        mask = (df[col] == 0) | (df[col].isna())
        prev_val = df.groupby('cell')[col].shift(1)
        next_val = df.groupby('cell')[col].shift(-1)

        fill_val = pd.Series(index=df.index, dtype=float)

        both = mask & prev_val.notna() & next_val.notna()
        fill_val[both] = (prev_val[both] + next_val[both]) / 2

        df.loc[mask, col] = fill_val[mask]

    return df

# clean the family IDS
before = final_ots["tracked_family_id"]
final_ots = clean_family_ids_temporal(final_ots)
after = final_ots["tracked_family_id_clean"]

# Only compare indices that exist in both
common_idx = before.index.intersection(after.index)
changed = (before.loc[common_idx] != after.loc[common_idx]).sum()
print("STEP 1 — Clean tracked_family_id OTS")
print(f"Fixed family ID entries: {changed}\n")

#fill interpolated values 
before = final_ots[["ncells_ot", "min_tb_ot"]].isna().sum().sum()
final_ots = fill_interpolated_values_ot(final_ots)
after = final_ots[["ncells_ot", "min_tb_ot"]].isna().sum().sum()

print("STEP 2 — Fill interpolated values")
print(f"  Missing before: {before:,}")
print(f"  Missing after:  {after:,}\n")


# Filter cells with zero lifetime
ots_before = len(final_ots)
valid_ots_cells = (
    final_ots.groupby("cell")["time_cell"]
    .max()
    .loc[lambda x: x > pd.Timedelta(minutes=1)]
    .index
)
final_ots = final_ots[final_ots["cell"].isin(valid_ots_cells)]

print("STEP 3 — Remove zero-duration OT cells")
print(f"  Rows removed: {ots_before - len(final_ots):,}")
print(f"  Remaining rows: {len(final_ots):,}\n")

anv_before = len(final_anvils)
valid_anvil_cells = (
    final_anvils.groupby("cell")["time_cell"]
    .max()
    .loc[lambda x: x > pd.Timedelta(minutes=9)]
    .index
)
final_anvils = final_anvils[final_anvils["cell"].isin(valid_anvil_cells)]

print("STEP 3b — Remove zero-duration ANVIL cells")
print(f"  Rows removed: {anv_before - len(final_anvils):,}")
print(f"  Remaining rows: {len(final_anvils):,}\n")


fam_before = len(final_anvils)
valid_anvil_cells = (
    final_anvils.groupby("tracked_family_id")["time_family"]
    .max()
    .loc[lambda x: x > pd.Timedelta(minutes=9)]
    .index
)
final_anvils = final_anvils[final_anvils["tracked_family_id"].isin(valid_anvil_cells)]

print("STEP 3c — Remove zero-duration FAMILY cells")
print(f"  Rows removed: {fam_before - len(final_anvils):,}")
print(f"  Remaining rows: {len(final_anvils):,}\n")


# Filter cells with area <4000 (defined in feature identification)
fam_before = len(final_anvils)
valid_anvil_cells = (
    final_anvils
    .groupby("tracked_family_id")["ncells_family"]
    .max()
    .loc[lambda x: x >= 4000]
    .index
)

final_anvils = final_anvils[
    final_anvils["tracked_family_id"].isin(valid_anvil_cells)
]
print("STEP 4 — Remove zero-area FAMILY cells")
print(f"  Rows removed: {fam_before - len(final_anvils):,}")
print(f"  Remaining rows: {len(final_anvils):,}\n")

# consistency check
# Get all valid family IDs from anvils
valid_family_ids_from_anvils = set(final_anvils["tracked_family_id"].unique())

# Collect OT family IDs
ot_family_ids = set(final_ots["tracked_family_id_clean"].unique())

# Identify missing OT IDs in the anvil dataset
missing_in_anvils = ot_family_ids - valid_family_ids_from_anvils

print(f"Unique OT family IDs:   {len(ot_family_ids):,}")
print(f"Unique Anvil family IDs: {len(valid_family_ids_from_anvils):,}")

if len(missing_in_anvils) == 0:
    print("All OT family IDs appear in the Anvil dataset. GOOD.")
else:
    print("WARNING: Some OT family IDs do NOT appear in the Anvil dataset!")

    #remove the offending OT rows
    before_count = len(final_ots)
    final_ots = final_ots[final_ots["tracked_family_id_clean"].isin(valid_family_ids_from_anvils)]
    removed_count = before_count - len(final_ots)
    print(f"Removed OT rows with invalid family IDs: {removed_count:,}")


INITIAL ROW COUNTS
OTS rows:    6,695,499
Anvil rows:  1,103,800

STEP 1 — Clean tracked_family_id OTS
Fixed family ID entries: 11674



/tmp/ipykernel_469135/1175734909.py:86: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[209.8125232642767 207.01483213694928 209.54116822274773
 199.10723145224614 206.84455036863983 207.0669176506135]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  fill_val[both] = (prev_val[both] + next_val[both]) / 2


STEP 2 — Fill interpolated values
  Missing before: 63,641
  Missing after:  63,635

STEP 3 — Remove zero-duration OT cells
  Rows removed: 124
  Remaining rows: 1,877,913

STEP 3b — Remove zero-duration ANVIL cells
  Rows removed: 55,775
  Remaining rows: 1,048,025

STEP 3c — Remove zero-duration FAMILY cells
  Rows removed: 6,589
  Remaining rows: 1,041,436

STEP 4 — Remove zero-area FAMILY cells
  Rows removed: 0
  Remaining rows: 1,041,436

CONSISTENCY CHECK
Unique OT family IDs:   2,533
Unique Anvil family IDs: 6,220

❌ WARNING: Some OT family IDs do NOT appear in the Anvil dataset!
Removed OT rows with invalid family IDs: 3,582


In [80]:
# make a clean column
final_anvils['tracked_family_id_clean'] = final_anvils['tracked_family_id']

In [83]:
# calculate tropopause relative depth 
final_ots['trop_relative_depth'] = (final_ots['min_tb_ot'] - final_ots['trop_temp']) / -3.25 # K/km

In [84]:
final_ots.to_pickle(out_base_folder + f"/features/final_ots_matched_QC_{version}_{sector}_-20_{start_str}_{end_str}.p")
final_anvils.to_pickle(out_base_folder + f"/features/final_families_QC_{version}_{sector}_-20_{start_str}_{end_str}.p")
